<a href="https://colab.research.google.com/github/NaveenHerath/llm-from-scratch/blob/main/day4_bigram_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Neural Nets and Loss
A bigram language model written as a pytorch module that returns a loss.

In [20]:
#Imports
import math
import torch
import torch.nn as nn
from torch.nn import functional as F

In [1]:
#1.Load the text
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
#this reads the shakespeare file into a one single string


From the previous code, get the tokenizer and the batch. Get the input and output batch.

In [2]:
#2.Building the vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
#this sorts the unique charactes in the dataset and identify them

In [8]:
vocab_size

65

In [3]:
#3.Translators
stoi = {ch: i  for i, ch in enumerate(chars)} #string to integer
itos = {i: ch for i, ch in enumerate(chars)} #int to str

In [4]:
#4.Encode and Decode
encode = lambda s: [stoi[c] for c in s] #encoder
decode = lambda l: ''.join([itos[i] for i in l]) #decoder

In [5]:
#5.Turn the whole dataset into a tensor
import torch
data = torch.tensor(encode(text), dtype = torch.long)
#store those numbers in to a tensor

In [10]:
data.shape

torch.Size([1115394])

In [6]:
#6.Train/Validation split
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]
#so the one dimensional matrix/array is now partitioned into 2 sections

In [13]:
#7.Get batch
block_size = 8 #how many characters models sees at once
batch_size = 4 #how many chunks we process in parallel

#we are writing a function to run through the batches of blocks
def get_batch(split):
  data = train_data if split == 'train' else val_data #tells what part to look at
  ix = torch.randint(len(data) - block_size, (batch_size,)) #picks 4 random strating positions, starting position must be within 0 to (len-blocksize)
  xb = torch.stack([data[i:i+block_size] for i in ix]) #cuts out 8 characters starting at ach of those positions torch.ctack thenput those on top of eac otherr to make a 4*8 grid
  yb = torch.stack([data[i+1:i+block_size+1] for i in ix]) #this do the same thing shifted one text to the right to make them labels
  return xb, yb

#xb = x batch and yb =  y batch
#so we now we have xb, yb of 4*8 tensors which are consist of 4 random 8 character lines each while x has the random picks and the y has the labels.

In [18]:
# Call get_batch to define xb (inputs) and yb (targets/labels)
xb, yb = get_batch('train')

print("xb shape:", xb.shape)
print("yb shape:", yb.shape)
print("\nInputs (xb):\n", xb)
print("\nTargets (yb):\n", yb)

xb shape: torch.Size([4, 8])
yb shape: torch.Size([4, 8])

Inputs (xb):
 tensor([[43,  6,  0, 13, 52, 42,  1, 14],
        [49,  1, 46, 43,  1, 47, 57, 10],
        [52, 45, 56, 63,  0, 13, 52, 42],
        [ 0,  0, 13, 14, 20, 27, 30, 31]])

Targets (yb):
 tensor([[ 6,  0, 13, 52, 42,  1, 14, 53],
        [ 1, 46, 43,  1, 47, 57, 10,  1],
        [45, 56, 63,  0, 13, 52, 42,  1],
        [ 0, 13, 14, 20, 27, 30, 31, 27]])


Building the Model

In [28]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)      # (B, T, C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)              # (B*T, C)
            targets = targets.view(B*T)               # (B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)                  # (B, T, C)
            logits = logits[:, -1, :]                 # (B, C)
            probs = F.softmax(logits, dim=-1)         # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1)   # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1)   # (B, T+1)
        return idx

In [29]:
torch.manual_seed(1337)
model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb)
print(logits.shape)
print(loss)

torch.Size([32, 65])
tensor(4.7839, grad_fn=<NllLossBackward0>)


In [32]:
#generate 200 characters in the random weights model
start = torch.zeros((1, 1), dtype=torch.long)
out = model.generate(start, max_new_tokens=200)
print(decode(out[0].tolist()))


SBLBcHAUhk&PHdhcOb
nhJ?FJU?pRiOLQeUN!BxjPLiq-GJdUV'hsnla!murI!IM?SPNPq?VgC'R
pD3cLv-bxn-tL!upg
SZ!Uvdg CtxtT?hsiW:XxKIiPlagHIsr'zKSVxza?GlDWObPmRJgrIAcmspmZ&viCKot:u3qYXA:rZgv f:3Q-oiwUzqh'Z!I'zRS3SP 


In [31]:
print(-math.log(1/65))
print(loss.item())

4.174387269895637
4.783884048461914
